In [ ]:
!pip install https://github.com/kpu/kenlm/archive/master.zip
!pyctcdecode
!pip install datasets
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.6/553.6 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kenlm: filename=kenlm-0.2.0-cp312-cp312-linux_x86_64.whl size=3188043 sha256=d1bce4ab2a7c77a292e4bd46a9ae218ea7d78990d6eb69083e293d10cec4c677
  Stored in directory: /tmp/pip-ephem-wheel-cache-wrxhgwdk/wheels/92/c8/12/56d187154e078f0eaa74d059017fc1afe1c4d91fbce02ce8d9
Successfully built kenlm
/bin/bash: line 1: pyctcdecode: command not found


In [ ]:
!pip install pyctcdecode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 538.1/538.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 34.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 

Let's load a small excerpt of the [Librispeech dataset](https://huggingface.co/datasets/librispeech_asr) to demonstrate Wav2Vec2's speech transcription capabilities.

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk(<path>)
dataset

Loading dataset from disk:   0%|          | 0/31 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 6996
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 500
    })
})

In [ ]:
import IPython.display as ipd

audio_sample = dataset["test"][2]
print(audio_sample["sentence"])
ipd.Audio(data=audio_sample["audio"]["array"], autoplay=True, rate=audio_sample["audio"]["sampling_rate"])

मिद् सए ओड़ः रः हतु तनः।


Having chosen a data sample, we now load the fine-tuned model and processor.

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
path = <xlsr_model_path>
processor = Wav2Vec2Processor.from_pretrained(path)
model = Wav2Vec2ForCTC.from_pretrained(path)

Loading weights:   0%|          | 0/808 [00:04<?, ?it/s]

Next, we process the data

In [ ]:
inputs = processor(audio_sample["audio"]["array"], sampling_rate=16000, return_tensors="pt")

forward it to the model

In [ ]:
import torch

with torch.no_grad():
  logits = model(**inputs).logits

and decode it

In [ ]:
#no ST
predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)

transcription[0]

'नों मेलो से हमारा देह नीं मां केतरो मोटो धीरत से एक तीन रंग वालो झण्डों देखाड़ दे दों ।'

In [ ]:
from transformers import Wav2Vec2ProcessorWithLM

processor = Wav2Vec2ProcessorWithLM.from_pretrained("/content/with_LM")

In [ ]:
logits.shape

torch.Size([1, 470, 74])

In [ ]:
" ".join(sorted(processor.tokenizer.get_vocab()))

'</s> <pad> <s> <unk> | ँ ं ः अ आ इ ई उ ऊ ऋ ऍ ऎ ए ऐ ऑ ओ औ क ख ग घ ङ च छ ज झ ञ ट ठ ड ढ ण त थ द ध न ऩ प फ ब भ म य र ऱ ल ळ व श ष स ह ़ ा ि ी ु ू ृ ॅ े ै ॉ ो ौ ् । ॥'

In [ ]:
transcription = processor.batch_decode(logits.numpy()).text
transcription[0]

'हमारी सोरियों आपडी परम्परा इज्जत लाम्बी कर देदी अने नवी जगा मा बयरा साहाब हारू हमेशा कमीशन नी सुविधा सुरू कर देदी'

## **3. Build an *n-gram* with KenLM**

While large language models based on the [Transformer architecture](https://jalammar.github.io/illustrated-transformer/) have become the standard in NLP, it is still very common to use an ***n-gram*** LM to boost speech recognition systems - as shown in Section 1.

Looking again at Table 9 of Appendix C of the [official Wav2Vec2 paper](https://arxiv.org/abs/2006.11477), it can be noticed that using a *Transformer*-based LM for decoding clearly yields better results than using an *n-gram* model, but the difference between *n-gram* and *Transformer*-based LM is much less significant than the difference between *n-gram* and no LM.

*E.g.*, for the large Wav2Vec2 checkpoint that was fine-tuned on 10min only, an *n-gram* reduces the word error rate (WER) compared to no LM by *ca.* 80% while a *Transformer*-based LM *only* reduces the WER by another 23% compared to the *n-gram*. This relative WER reduction becomes less, the more data the acoustic model has been trained on. *E.g.*, for the large checkpoint a *Transformer*-based LM reduces the WER by merely 8% compared to an *n-gram* LM whereas the *n-gram* still yields a 21% WER reduction compared to no language model.

The reason why an *n-gram* is preferred over a *Transformer*-based LM is that *n-grams* come at a significantly smaller computational cost. For an *n-gram*, retrieving the probability of a word given previous words is almost only as computationally expensive as querying a look-up table or tree-like data storage - *i.e.* it's very fast compared to modern *Transformer*-based language models that would require a full forward pass to retrieve the next word probabilities.

For more information on how *n-grams* function and why they are (still) so useful for speech recognition, the reader is advised to take a look at [this excellent summary](https://web.stanford.edu/~jurafsky/slp3/3.pdf) from Stanford.

Great, let's see step-by-step how to build an *n-gram*. We will use the popular [KenLM library](https://github.com/kpu/kenlm) to do so. Let's start by installing the Ubuntu library prerequisites:

In [ ]:
!sudo apt install build-essential cmake libboost-system-dev libboost-thread-dev libboost-program-options-dev libboost-test-dev libeigen3-dev zlib1g-dev libbz2-dev liblzma-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libbz2-dev is already the newest version (1.0.8-5build1).
libbz2-dev set to manually installed.
liblzma-dev is already the newest version (5.2.5-2ubuntu1).
liblzma-dev set to manually installed.
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
zlib1g-dev is already the newest version (1:1.2.11.dfsg-2ubuntu9.2).
zlib1g-dev set to manually installed.
The following additional packages will be installed:
  libboost-atomic1.74-dev libboost-atomic1.74.0 libboost-chrono1.74-dev
  libboost-chrono1.74.0 libboost-date-time1.74-dev libboost-date-time1.74.0
  libboost-program-options1.74-dev libboost-program-options1.74.0
  libboost-serialization1.74-dev libboost-serialization1.74.0
  libboost-system1.74-dev libboost-system1.74.0 libboost-test1.74-dev
  libboost-test1.74.0 libboost-thread1.74-dev libboost-thread1.74.0
Suggested 

before downloading and unpacking the KenLM repo.

In [ ]:
!wget -O - https://kheafield.com/code/kenlm.tar.gz | tar xz

--2026-05-22 20:46:21--  https://kheafield.com/code/kenlm.tar.gz
Resolving kheafield.com (kheafield.com)... 129.80.89.152, 2603:c020:4009:8710:ca:11:17:0
Connecting to kheafield.com (kheafield.com)|129.80.89.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 491888 (480K) [application/octet-stream]
Saving to: ‘STDOUT’

-                   100%[===================>] 480.36K  1.87MB/s    in 0.3s    

2026-05-22 20:46:22 (1.87 MB/s) - written to stdout [491888/491888]



KenLM is written in C++, so we'll make use of `cmake` to build the binaries.

In [ ]:
!mkdir kenlm/build && cd kenlm/build && cmake .. && make -j2
!ls kenlm/build/bin

CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:85 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "

Great, as we can see, the executable functions have successfully been built under `kenlm/build/bin/`.

KenLM by default computes an *n-gram* with [Kneser-Ney smooting](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing). All text data used to create the *n-gram* is expected to be stored in a text file.
We download our dataset and save it as a `.txt` file.

Now, we just have to run KenLM's `lmplz` command to build our *n-gram*, called `"5gram.arpa"`. As it's relatively common in speech recognition, we build a *5-gram* by passing the `-o 5` parameter.
For more information on the different *n-gram* LM that can be built
with KenLM, one can take a look at the [official website of KenLM](https://kheafield.com/code/kenlm/).

Executing the command below might take a minute or so.

In [ ]:

!kenlm/build/bin/lmplz -o 5 <"mundari_ASR_text" > "5gram.arpa"

=== 1/5 Counting and sorting n-grams ===
Reading /content/mundari_ASR.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 95252 types 19361
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:232332 2:1061895744 3:1991054720 4:3185687296 5:4645794304
Statistics:
1 19361 D1=0.735441 D2=1.10544 D3+=1.17775
2 71417 D1=0.867892 D2=1.23517 D3+=1.24698
3 88261 D1=0.956479 D2=1.44382 D3+=1.37627
4 84522 D1=0.986793 D2=1.60423 D3+=2.05268
5 76193 D1=0.989942 D2=1.89201 D3+=2.43432
Memory estimate for binary LM:
type      kB
probing 7554 assuming -p 1.5
probing 9060 assuming -r models -p 1.5
trie    3713 without quantization
trie    2105 assuming -q 8 -b 8 quantization 
trie    3414 assuming -a 22 array pointer compression
trie    1806 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Cal

Great, we have built a *5-gram* LM! Let's inspect the first couple of lines.

In [ ]:
!head -20 5gram.arpa

\data\
ngram 1=30479
ngram 2=117105
ngram 3=150995
ngram 4=149965
ngram 5=141429

\1-grams:
-5.1200976	<unk>	0
0	<s>	-0.70344514
-1.6979618	</s>	0
-4.798187	दनियानें	-0.0712509
-3.4606547	मालम	-0.19803612
-1.6766628	नी	-0.3037243
-3.9059079	हतुं	-0.24605289
-2.6315491	की	-0.1516727
-2.5892255	भारत	-0.4915985
-3.252046	कन	-0.10960483
-5.0045676	पैदाइसी	-0.0712509
-3.1679711	ताकत	-0.19169705


Great, this looks better! We're done at this point and all that is left to do is to correctly integrate the `"ngram"` with [`pyctcdecode`](https://github.com/kensho-technologies/pyctcdecode) and 🤗 Transformers.

## **4. Combine an *n-gram* with Wav2Vec2**

In a final step, we want to wrap the *5-gram* into a `Wav2Vec2ProcessorWithLM` object to make the *5-gram* boosted decoding as seamless as shown in Section 1.
We start by downloading the currently "LM-less" processor of [`xls-r-300m-sv`](https://huggingface.co/hf-test/xls-r-300m-sv).

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(<path_model>)

Next, we extract the vocabulary of its tokenizer as it represents the `"labels"` of `pyctcdecode`'s `BeamSearchDecoder` class.

In [ ]:
vocab_dict = processor.tokenizer.get_vocab()
sorted_vocab_dict = {k.lower(): v for k, v in sorted(vocab_dict.items(), key=lambda item: item[1])}

The `"labels"` and the previously built `5gram_correct.arpa` file is all that's needed to build the decoder.

In [ ]:
from pyctcdecode import build_ctcdecoder

decoder = build_ctcdecoder(
    labels=list(sorted_vocab_dict.keys()),
    kenlm_model_path="5gram.arpa",
)

We can safely ignore the warning and all that is left to do now is to wrap the just created `decoder`, together with the processor's `tokenizer` and `feature_extractor` into a `Wav2Vec2ProcessorWithLM` class.

In [ ]:
from transformers import Wav2Vec2ProcessorWithLM

processor_with_lm = Wav2Vec2ProcessorWithLM(
    feature_extractor=processor.feature_extractor,
    tokenizer=processor.tokenizer,
    decoder=decoder
)

We want to directly upload the LM-boosted processor into
the model folder of [`xls-r-300m-sv`](https://huggingface.co/hf-test/xls-r-300m-sv) to have all relevant files in one place.

Let's clone the repo, add the new decoder files and upload them afterward.
First, we need to install `git-lfs`.

In [ ]:
processor_with_lm.save_pretrained("bhili_LM")

Let's inspect the local repository. The `tree` command conveniently can also show the size of the different files.

In [ ]:
!tree -h bhili_LM/

/bin/bash: line 1: tree: command not found


As can be seen the *5-gram* LM is quite large - it amounts to more than 4 GB.
To reduce the size of the *n-gram* and make loading faster, `kenLM` allows converting `.arpa` files to binary ones using the `build_binary` executable.

Let's make use of it here.

In [ ]:
!kenlm/build/bin/build_binary bhili_LM/language_model/5gram.arpa bhili_LM/language_model/5gram.bin

Reading bhili_LM/language_model/5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
SUCCESS


Great, it worked! Let's remove the `.arpa` file and check the size of the binary *5-gram* LM.

In [ ]:
!rm bhili_LM/language_model/5gram.arpa && tree -h bhili_LM/

/bin/bash: line 1: tree: command not found


Nice, we reduced the *n-gram* by more than half to less than 2GB now. In the final step, let's upload all files.

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.1 MB/s eta 0:00:00


In [ ]:
from transformers import Wav2Vec2ProcessorWithLM, Wav2Vec2ForCTC
from datasets import load_from_disk
import torch
from jiwer import wer
from tqdm import tqdm



In [ ]:
processor_path = "bhili_LM"
processor = Wav2Vec2ProcessorWithLM.from_pretrained(processor_path)

In [ ]:

# processor with LM
processor = Wav2Vec2ProcessorWithLM.from_pretrained(processor_path)


In [ ]:
#finetuned model
model_path = <path>
model = Wav2Vec2ForCTC.from_pretrained(model_path)

Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

In [ ]:
import torch
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2ProcessorWithLM
from jiwer import wer
from jiwer import cer
from tqdm import tqdm

# ---- Load dataset ----
# example:
# dataset = load_dataset("your_dataset_name", split="test")

test_dataset = dataset["test"]  # or dataset["test"]

# ---- Load processor + model ----


model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

device = model.device

# ---- WER computation ----
all_preds = []
all_refs = []

for sample in tqdm(test_dataset):
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    ref_text = sample["sentence"].lower().strip()

    inputs = processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    # LM-based decoding
    #pred_text = processor.batch_decode(logits.cpu().numpy())[0]
    pred_text = processor.batch_decode(logits.cpu().numpy()).text[0]
    pred_text = pred_text.strip()

    all_preds.append(pred_text)
    all_refs.append(ref_text)

# ---- Final WER ----
wer_score = wer(all_refs, all_preds)
cer_score = cer(all_refs, all_preds)
print(f"CER: {cer_score}")
print("WER:", wer_score)


In [ ]:
import json

results = []

for i, (ref, pred) in enumerate(zip(all_refs, all_preds)):
    results.append({
        "id": i,
        "reference": ref,
        "prediction": pred
    })

with open("asr_predictions_mundari_stage_5.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saved to asr_predictions.json")


Saved to asr_predictions.json


#Inference

In [ ]:
"""
Production ASR Pipeline with Language Model
Supports WAV and MP3 audio formats
"""

import torch
import librosa
import numpy as np
from transformers import Wav2Vec2ForCTC, Wav2Vec2ProcessorWithLM
from pathlib import Path
from typing import Union, List, Dict
import warnings
warnings.filterwarnings('ignore')


class ASRPipeline:
    """
    Production-ready ASR pipeline with language model decoding.

    Args:
        model_path: Path to the trained Wav2Vec2ForCTC model
        processor_path: Path to the Wav2Vec2ProcessorWithLM
        device: Device to run inference on ('cuda' or 'cpu')
    """

    def __init__(
        self,
        model_path: str,
        processor_path: str,
        device: str = None
    ):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        print(f"Loading model from {model_path}...")
        self.model = Wav2Vec2ForCTC.from_pretrained(model_path)
        self.model.eval()
        self.model.to(self.device)

        print(f"Loading processor from {processor_path}...")
        self.processor = Wav2Vec2ProcessorWithLM.from_pretrained(processor_path)

        print(f"Pipeline ready on {self.device}")

    def load_audio(
        self,
        audio_path: Union[str, Path],
        target_sr: int = 16000
    ) -> np.ndarray:
        """
        Load audio file and resample to target sampling rate.

        Args:
            audio_path: Path to audio file (wav, mp3, etc.)
            target_sr: Target sampling rate (default: 16000 Hz)

        Returns:
            Audio array resampled to target_sr
        """
        audio_path = Path(audio_path)

        if not audio_path.exists():
            raise FileNotFoundError(f"Audio file not found: {audio_path}")

        # Load audio with librosa (handles wav, mp3, flac, etc.)
        audio, sr = librosa.load(str(audio_path), sr=target_sr, mono=True)

        return audio

    def transcribe(
        self,
        audio_path: Union[str, Path],
        return_confidence: bool = False
    ) -> Union[str, Dict]:
        """
        Transcribe a single audio file.

        Args:
            audio_path: Path to audio file
            return_confidence: If True, return dict with text and confidence scores

        Returns:
            Transcription text or dict with text and metadata
        """
        # Load audio
        audio = self.load_audio(audio_path)

        # Prepare inputs
        inputs = self.processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Inference
        with torch.no_grad():
            logits = self.model(**inputs).logits

        # Decode with language model
        decoded = self.processor.batch_decode(logits.cpu().numpy())
        transcription = decoded.text[0].strip()

        if return_confidence:
            return {
                'text': transcription,
                'audio_path': str(audio_path),
                'duration': len(audio) / 16000
            }

        return transcription

    def transcribe_batch(
        self,
        audio_paths: List[Union[str, Path]],
        batch_size: int = 8
    ) -> List[Dict]:
        """
        Transcribe multiple audio files in batches.

        Args:
            audio_paths: List of paths to audio files
            batch_size: Number of files to process at once

        Returns:
            List of dicts with transcription results
        """
        results = []

        for i in range(0, len(audio_paths), batch_size):
            batch_paths = audio_paths[i:i + batch_size]

            for audio_path in batch_paths:
                try:
                    result = self.transcribe(audio_path, return_confidence=True)
                    results.append(result)
                except Exception as e:
                    results.append({
                        'text': '',
                        'audio_path': str(audio_path),
                        'error': str(e)
                    })

        return results

    def transcribe_from_array(
        self,
        audio_array: np.ndarray,
        sampling_rate: int = 16000
    ) -> str:
        """
        Transcribe from numpy array (for streaming or pre-loaded audio).

        Args:
            audio_array: Audio as numpy array
            sampling_rate: Sampling rate of the audio

        Returns:
            Transcription text
        """
        # Resample if needed
        if sampling_rate != 16000:
            audio_array = librosa.resample(
                audio_array,
                orig_sr=sampling_rate,
                target_sr=16000
            )

        # Prepare inputs
        inputs = self.processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Inference
        with torch.no_grad():
            logits = self.model(**inputs).logits

        # Decode
        decoded = self.processor.batch_decode(logits.cpu().numpy())
        transcription = decoded.text[0].strip()

        return transcription

